In [2]:
!pip install pandas langchain



In [3]:
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter

# === 1. Load the filtered complaints CSV ===
df = pd.read_csv("/content/filtered.csv")

# === 2. Drop missing or empty narratives ===
df = df[df['Consumer complaint narrative'].notna()]
df = df[df['Consumer complaint narrative'].str.strip() != ""]

# === 3. Set up the chunker ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Max characters per chunk
    chunk_overlap=50     # Overlap between chunks
)

# === 4. Apply chunking ===
chunked_rows = []
for idx, row in df.iterrows():
    complaint_id = row["Complaint ID"]
    product = row["Product"]
    narrative = row["Consumer complaint narrative"]

    # Split text into chunks
    chunks = text_splitter.split_text(narrative)

    for i, chunk in enumerate(chunks):
        chunked_rows.append({
            "Complaint ID": complaint_id,
            "Product": product,
            "Chunk Number": i + 1,
            "Narrative Chunk": chunk
        })

# === 5. Create a new DataFrame of chunks ===
chunk_df = pd.DataFrame(chunked_rows)

# === 6. Save it for embedding step ===
chunk_df.to_csv("/content/complaint_chunks.csv", index=False)
print("✅ Chunking complete. Saved to 'complaint_chunks.csv'")



✅ Chunking complete. Saved to 'complaint_chunks.csv'


In [1]:
!pip install sentence-transformers pandas


In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# === 1. Load your chunked complaint data ===
df = pd.read_csv("/content/complaint_chunks.csv")

# === 2. Load the MiniLM model ===
model = SentenceTransformer('all-MiniLM-L6-v2')

# === 3. Embed all text chunks ===
texts = df['Narrative Chunk'].fillna("").tolist()
embeddings = model.encode(texts, show_progress_bar=True)

# === 4. Save the embeddings separately ===
np.save("/content/embeddings.npy", embeddings)  # Save for FAISS
print("✅ Saved embeddings to 'embeddings.npy'")

# === Optional: Save combined CSV (text + vector) ===
# WARNING: Very large file if you save all dimensions
embedding_df = pd.DataFrame(embeddings)
final_df = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)
final_df.to_csv("/content/complaint_chunks_with_embeddings.csv", index=False)
print("✅ Saved to 'complaint_chunks_with_embeddings.csv'")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2121 [00:00<?, ?it/s]

✅ Saved embeddings to 'embeddings.npy'
✅ Saved to 'complaint_chunks_with_embeddings.csv'


In [ ]:
pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 23.0 MB/s eta 0:00:00


In [6]:
import faiss
import numpy as np

# Load embeddings
embeddings = np.load("embeddings.npy").astype('float32')

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save index
faiss.write_index(index, "/content/faiss_index.index")
print("✅ FAISS index saved at /content/faiss_index.index")


✅ FAISS index saved at /content/faiss_index.index


In [7]:
# === Load index ===
index = faiss.read_index("/content/faiss_index.index")

# === Embed the query ===
query = "They charged me fees I didn’t agree to."
query_vec = model.encode([query]).astype('float32')

# === Search for top 5 matches ===
D, I = index.search(query_vec, k=5)

# === Show matching complaints ===
for rank, idx in enumerate(I[0]):
    print(f"\nRank {rank+1}:")
    print(df.iloc[idx][['Complaint ID', 'Product', 'Narrative Chunk']])



Rank 1:
Complaint ID                                              12974654.0
Product                                  checking or savings account
Narrative Chunk    they charged me overdraft and account fees tha...
Name: 25543, dtype: object

Rank 2:
Complaint ID                                               2918962.0
Product                                  checking or savings account
Narrative Chunk    and charged more overdraft fees on top they re...
Name: 30332, dtype: object

Rank 3:
Complaint ID                                              12138217.0
Product                                                  credit card
Narrative Chunk    notified me about their decision but they kept...
Name: 15919, dtype: object

Rank 4:
Complaint ID                                              12628591.0
Product                                                  credit card
Narrative Chunk    i was charge an anual fee for 4900 on xxxx2025...
Name: 67385, dtype: object

Rank 5:
Complaint ID       

In [13]:
#chromadb
!pip install --upgrade chromadb




In [9]:
#chromaDb

In [15]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

# === Load data and embeddings ===
df = pd.read_csv("/content/complaint_chunks.csv")
embeddings = np.load("/content/embeddings.npy")
texts = df['Narrative Chunk'].fillna("").tolist()
metadata = df[['Complaint ID', 'Product']].astype(str).to_dict('records')
ids = [str(i) for i in df.index]

# === Setup ChromaDB ===
client = PersistentClient(path="/content/chroma/")
collection = client.get_or_create_collection("complaints")

# === Add in batches (max 5000 per batch for safety) ===
batch_size = 5000
for i in range(0, len(texts), batch_size):
    end = i + batch_size
    collection.add(
        documents=texts[i:end],
        embeddings=embeddings[i:end].tolist(),
        metadatas=metadata[i:end],
        ids=ids[i:end]
    )
    print(f"✅ Added batch {i}–{end}")

print("✅ All data added to ChromaDB successfully.")


✅ Added batch 0–5000
✅ Added batch 5000–10000
✅ Added batch 10000–15000
✅ Added batch 15000–20000
✅ Added batch 20000–25000
✅ Added batch 25000–30000
✅ Added batch 30000–35000
✅ Added batch 35000–40000
✅ Added batch 40000–45000
✅ Added batch 45000–50000
✅ Added batch 50000–55000
✅ Added batch 55000–60000
✅ Added batch 60000–65000
✅ Added batch 65000–70000
✅ All data added to ChromaDB successfully.


In [16]:
from sentence_transformers import SentenceTransformer

# === Step 1: Load the ChromaDB collection ===
client = PersistentClient(path="/content/chroma/")
collection = client.get_collection("complaints")

# === Step 2: Define your test query ===
query_text = "I was charged a hidden fee on my personal loan"

# === Step 3: Embed the query using the same model ===
model = SentenceTransformer('all-MiniLM-L6-v2')
query_embedding = model.encode([query_text]).tolist()

# === Step 4: Perform the search ===
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5  # You can change this to 10, 20, etc.
)

# === Step 5: Display results ===
print("\n🔎 Top matching complaints:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
    print(f"\nRank {i}:")
    print("🆔 Complaint ID:", meta.get("Complaint ID"))
    print("📦 Product:", meta.get("Product"))
    print("💬 Narrative:", doc[:500], "..." if len(doc) > 500 else "")



🔎 Top matching complaints:

Rank 1:
🆔 Complaint ID: 11830082.0
📦 Product: payday loan, title loan, personal loan, or advance loan
💬 Narrative: on xxxxyear i applied for a loan of 30000 through xxxx xxxx xxxx during the loan application process the company explicitly stated the terms of the loan including the fees and repayment structure however after accepting the loan i was unexpectedly charged an additional hidden fee of 43000 which was not disclosed at the time of application i believe this fee is unfair deceptive and a violation of my consumer rights i attempted to resolve this matter with the company directly but they refused to 

Rank 2:
🆔 Complaint ID: 12962614.0
📦 Product: payday loan, title loan, personal loan, or advance loan
💬 Narrative: took out a small loan from this company and was charged fees and interest that i didnt know about 

Rank 3:
🆔 Complaint ID: 12408858.0
📦 Product: credit card
💬 Narrative: and therefore was not given an opportunity to include that amount in 

In [ ]:
#building RAG with LLM

In [17]:
!pip install transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 8.0 MB/s eta 0:00:00


In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# === 1. Example context and question (replace with your retrieval results) ===
context = """
Complaint 1: I was charged hidden fees on my personal loan.
Complaint 2: I didn’t receive the service I paid for on Buy Now Pay Later.
Complaint 3: Unexpected deductions from my savings account.
"""
user_question = "Why are users upset about personal loans?"

# === 2. Load flan-t5-base model and tokenizer ===
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# === 3. Prepare prompt by combining context + question ===
prompt = f"Answer the question based on the context:\nContext: {context}\nQuestion: {user_question}"

# === 4. Tokenize inputs ===
inputs = tokenizer(prompt, return_tensors="pt")

# === 5. Generate answer ===
outputs = model.generate(**inputs, max_new_tokens=100)

# === 6. Decode generated tokens ===
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated answer:", answer)

# === 7. Save answer to file ===
with open("/content/answers.txt", "w") as f:
    f.write("Q: " + user_question + "\n\n")
    f.write("A:\n" + answer)

print("✅ Saved answer to /content/answers.txt")


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated answer: hidden fees
✅ Saved answer to /content/answers.txt


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gradio as gr

# Load the flan-t5-base model
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Load the context from file or hardcoded for now
with open("/content/answers.txt", "r") as f:
    file_content = f.read()

# Extract the context only (optional improvement)
# If your file only has one answer, you can load a fixed chunk instead
context = """
Complaint 1: I was charged hidden fees on my personal loan.
Complaint 2: I didn’t receive the service I paid for on Buy Now Pay Later.
Complaint 3: Unexpected deductions from my savings account.
"""  # You can replace this with FAISS/Chroma retrieved content

# Define the function to run generation
def generate_answer(user_question):
    prompt = f"Answer the question based on the context:\nContext: {context}\nQuestion: {user_question}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=150)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# Create Gradio UI
iface = gr.Interface(
    fn=generate_answer,
    inputs=gr.Textbox(label="Ask your question"),
    outputs=gr.Textbox(label="Answer"),
    title="💬 Complaint Q&A Assistant",
    description="Ask a question based on complaint data. The answer is generated by FLAN-T5."
)

iface.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4867d6a9277e105da8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# === 1. Load data and FAISS index ===
chunk_df = pd.read_csv("complaint_chunks.csv")  # has 'Narrative Chunk' column
faiss_index = faiss.read_index("faiss_index.index")  # prebuilt FAISS index

# === 2. Load embedding model ===
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# === 3. Load FLAN-T5 model ===
llm_model = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(llm_model)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model)

# === 4. Define: Retrieve top-k complaint chunks ===
def retrieve_top_k_chunks(user_question, k=5):
    query_vec = embed_model.encode([user_question]).astype("float32")
    D, I = faiss_index.search(query_vec, k)
    return chunk_df.iloc[I[0]]["Narrative Chunk"].tolist()

# === 5. Define: Build prompt and generate answer ===
def generate_answer(context_chunks, question):
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a financial complaints assistant.
Use only the context below to answer the question.
If you don't know the answer, say "I don't know."

Context:
{context}

Question: {question}
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# === 6. Run a test question ===
user_question = "Why are users upset about personal loans?"
top_chunks = retrieve_top_k_chunks(user_question)
answer = generate_answer(top_chunks, user_question)

# === 7. Save answer to file ===
with open("/content/answers.txt", "w") as f:
    f.write("Q: " + user_question + "\n\n")
    f.write("Context:\n" + "\n".join(top_chunks) + "\n\n")
    f.write("A:\n" + answer)

print("✅ Answer saved to /content/answers.txt")
print("\nGenerated Answer:\n", answer)


✅ Answer saved to /content/answers.txt

Generated Answer:
 They are confusing and designed to cause consumers to pay a surprise enormous interest fee


In [25]:
import gradio as gr

# === Gradio function: Runs the full pipeline ===
def rag_qa(user_question):
    try:
        top_chunks = retrieve_top_k_chunks(user_question)
        answer = generate_answer(top_chunks, user_question)
        context_preview = "\n\n".join(top_chunks[:3])  # Show top 3 chunks
        return f"📌 **Question:** {user_question}\n\n" + \
               f"🧠 **Answer:**\n{answer}\n\n" + \
               f"📚 **Context Snippets:**\n{context_preview}"
    except Exception as e:
        return f"❌ Error: {str(e)}"

# === Gradio UI setup ===
gr.Interface(
    fn=rag_qa,
    inputs=gr.Textbox(lines=2, placeholder="Ask your question about complaints...", label="💬 Your Question"),
    outputs=gr.Markdown(label="🧾 Answer"),
    title="📊 Complaint Q&A Assistant",
    description="Ask a question and get answers from real customer complaint data using RAG (Retriever + Generator).",
    theme="default"
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88c4802e262d50a622.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
